# Multimodal Meme Understanding: Image, Text, or Both?
### Deep Learning Final Project
Dataset: Memotion 7k (SemEval-2020 Task 8, Task A: 3-class sentiment).

Pipeline (see `docs/project-roadmap.md`):
1. Environment setup & Google Drive mount
2. Dataset download & verification
3. Data preparation & leakage control (Hold-out split)
4. Text-only Baseline (BERT) - Role B
5. Image-only Baseline (ResNet50) - Role C
6. Multimodal Fusion (Concat & Cross-Attention) - Role D
7. Evaluation, ablation & error analysis - Role E & F
8. Demo inference

## 1. Environment Setup & Google Drive Mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

BRANCH = os.environ.get('BRANCH', 'role/d-fusion')
REPO_URL = 'https://github.com/arcrek/dl2026.git'
ROOT = '/content/drive/MyDrive/dl2026_memotion'
os.environ['DATA_DIR'] = '/content/data/memotion'
os.environ['FEATURE_DIR'] = f'{ROOT}/features'
os.environ['RESULTS_DIR'] = f'{ROOT}/results'

for d in ('features', 'results'):
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)

if os.path.exists('/content/repo'):
    !git -C /content/repo fetch && git -C /content/repo checkout $BRANCH && git -C /content/repo pull
else:
    !git clone -b $BRANCH $REPO_URL /content/repo
%cd /content/repo

!pip install -q transformers scikit-learn pyarrow huggingface_hub pytest pandas torchvision


## 2. Dataset Download & Verification
> **Note:** For the Text-only Baseline (Role B), you can pass `--jsonl-only` to `scripts/download_data.sh` to download text metadata in seconds without fetching 1.5GB of images.

In [ ]:
TAR = f'{ROOT}/memotion.tar'
if os.path.exists(TAR) and not os.path.exists('/content/data/memotion/train.jsonl'):
    !mkdir -p /content/data && tar -xf "$TAR" -C /content/data

!bash scripts/download_data.sh
!python src/data.py --images

if not os.path.exists(TAR):
    !tar -cf "$TAR" -C /content/data memotion


## 3. Data Preparation & Preprocessing

### Label Normalization
Raw sentiment labels in Memotion 7k follow a 5-point scale:
`very_positive`, `positive`, `neutral`, `negative`, `very_negative`

For standard 3-class sentiment classification, they are consolidated into:
- `positive` (from very positive, positive, pos)
- `neutral` (from neutral, neu)
- `negative` (from very negative, negative, neg)


In [ ]:
import os, sys
sys.path.insert(0, "src")
import pandas as pd
from PIL import Image
from preprocessing import (
    normalize_sentiment,
    is_valid_image,
    get_image_path,
    load_raw_memotion_df,
    LABEL2ID,
    ID2LABEL,
)

def normalize_sentiment(label):
    if pd.isna(label):
        return None

    label = str(label).strip().lower()
    label = label.replace("_", " ")

    if label in ["very positive", "positive", "pos"]:
        return "positive"
    if label in ["neutral", "neu"]:
        return "neutral"
    if label in ["very negative", "negative", "neg"]:
        return "negative"

    return None


### 3.1. Create Clean DataFrame
After this step, the dataset contains only 3 primary columns:
`text`, `image_path`, `label_name`

Samples missing valid image files or sentiment labels are dropped.


In [ ]:
# Load DataFrame (supports Kaggle df, Colab, or local jsonl)
if "df" not in locals():
    df = load_raw_memotion_df(os.environ.get("DATA_DIR", "data/memotion"))

data = pd.DataFrame()
text_col = "text_ocr" if "text_ocr" in df.columns else ("text_corrected" if "text_corrected" in df.columns else "text")
image_col = "image_name" if "image_name" in df.columns else ("img" if "img" in df.columns else "image")
sentiment_col = "overall_sentiment" if "overall_sentiment" in df.columns else ("raw_sentiment" if "raw_sentiment" in df.columns else "label")

data["text"] = df[text_col].fillna("").astype(str)
data["image_path"] = df[image_col].apply(lambda x: get_image_path(x, base_dir=os.environ.get("DATA_DIR", "data/memotion")))
data["label_name"] = df[sentiment_col].apply(normalize_sentiment)

print("Before filtering invalid samples:", len(data))
print("Samples with found images:", data["image_path"].notna().sum())
print("Samples with valid labels:", data["label_name"].notna().sum())

# Drop samples missing image or label
data = data.dropna(subset=["image_path", "label_name"]).reset_index(drop=True)

print("After dropping missing image/label:", len(data))
print(data["label_name"].value_counts())
data.head()


### 3.2. Filter Corrupted Images Before Training
Some images in Memotion 7k may be corrupted or truncated. If not filtered beforehand, the DataLoader may crash during training with:
```
OSError: image file is truncated
```
Therefore, each image file is verified once before training.


In [ ]:
def is_valid_image(path):
    try:
        with Image.open(path) as img:
            img.verify()

        with Image.open(path) as img:
            img.convert("RGB")

        return True
    except Exception:
        return False

before_filter = len(data)
data = data[data["image_path"].apply(is_valid_image)].reset_index(drop=True)
after_filter = len(data)

print("Corrupted images dropped:", before_filter - after_filter)
print("Final sample count for training:", after_filter)
print(data["label_name"].value_counts())
data.head()


### 3.3. Encode Labels to Numeric IDs
PyTorch models require numeric target labels rather than strings:
- `negative`: 0
- `neutral`: 1
- `positive`: 2


In [ ]:
label2id = {
    "negative": 0,
    "neutral": 1,
    "positive": 2,
}

id2label = {
    0: "negative",
    1: "neutral",
    2: "positive",
}

data["label"] = data["label_name"].map(label2id)
data.head()


## 4. Text-only Baseline (BERT) - Role B
Fine-tune `bert-base-uncased` with Differential Learning Rate (`1.5e-5` for backbone, `5e-4` for head) across seeds 0, 1, 2. Model selection uses hold-out Macro-F1, and test predictions are exported to `results/text_seed{0,1,2}.json`.

In [ ]:
!python src/text.py --seeds 0 1 2 --epochs 5 --batch 32


In [ ]:
!pytest tests/test_results.py -v


In [ ]:
import os, sys
sys.path.insert(0, 'src')
import numpy as np
import pandas as pd
from results import load_results

records = []
for s in [0, 1, 2]:
    try:
        res = load_results('text', s)
    except FileNotFoundError:
        continue
    records.append({
        'Seed': s,
        'Best Epoch (Holdout)': res['extra'].get('best_epoch', 'N/A'),
        'Holdout Macro-F1': round(res['extra'].get('holdout_macro_f1', 0.0), 4),
        'Test Accuracy': round(res['accuracy'], 4),
        'Test Macro-F1': round(res['macro_f1'], 4)
    })

if records:
    df = pd.DataFrame(records)
    if len(records) > 1:
        mean_row = pd.DataFrame([{
            'Seed': 'Mean +/- Std',
            'Best Epoch (Holdout)': '-',
            'Holdout Macro-F1': f"{df['Holdout Macro-F1'].mean():.4f} +/- {df['Holdout Macro-F1'].std():.4f}",
            'Test Accuracy': f"{df['Test Accuracy'].mean():.4f} +/- {df['Test Accuracy'].std():.4f}",
            'Test Macro-F1': f"{df['Test Macro-F1'].mean():.4f} +/- {df['Test Macro-F1'].std():.4f}"
        }])
        summary_df = pd.concat([df, mean_row], ignore_index=True)
    else:
        summary_df = df
    display(summary_df)
else:
    print('No results found. Run training cell above first.')


In [ ]:
try:
    from google.colab import files
    for s in [0, 1, 2]:
        f_path = f"results/text_seed{s}.json"
        if os.path.exists(f_path):
            files.download(f_path)
except Exception:
    pass


## 5. Image-only Baseline (ResNet50)


## 6. Multimodal Fusion: Cross-Attention (Role D)
Huấn luyện kiến trúc **Cross-Attention Fusion (Branch 3B - `both_cross_attn`)** qua 3 random seeds [0, 1, 2]:
- **Text Stream:** Các token ẩn từ BERT (`bert-base-uncased`) ($L \times 768$) được chiếu tuyến tính về không gian $L \times 256$ đóng vai trò **Query**.
- **Image Stream:** Feature map không gian từ `layer4` của ResNet50 ($7 \times 7 \times 2048 \to 49 \times 2048$) được chiếu tuyến tính về $49 \times 256$ đóng vai trò **Key** và **Value**.
- **Cross-Attention:** Multi-Head Attention ($H=4$) cho phép text tokens tương tác trực tiếp với các vùng không gian thị giác của ảnh.
- **Residual & Normalization:** Residual connection $Q + \text{CrossAttn}(Q, K, V)$ kết hợp Layer Normalization.
- **Pooling & Head:** Trích xuất vector đại diện tại vị trí `[CLS]` token ($256$d) đưa qua MLP Classifier (Linear 256 -> 128 -> ReLU -> Dropout -> Linear 128 -> 3).
- **Chiến lược tối ưu:** Differential Learning Rate (`1.5e-5` cho backbone, `5e-4` cho fusion head), loss có trọng số `class_weights`, chọn mô hình tốt nhất theo holdout Macro-F1 và xuất kết quả test vào `results/both_cross_attn_seed{0,1,2}.json`.

*(Lưu ý: Phần Concat Fusion và Product Fusion do các thành viên khác trong nhóm phụ trách).*


### 6.1 Model Architecture & Loss/Optimizer Setup


In [ ]:
import sys
sys.path.insert(0, 'src')
import torch
import torch.nn as nn
from cross_attn import (
    CrossAttentionArgs,
    CrossAttentionFusionModel,
    MultimodalMemeDataset,
    build_loss_and_optimizer,
    make_multimodal_collate_fn,
    evaluate,
    train_epoch,
    train_loop,
    run_training,
)

print('Part 1: Cross-Attention architecture, loss & optimizer modules loaded.')


### 6.2 Evaluation Function


In [ ]:
from cross_attn import evaluate

print('Part 2: Multimodal evaluation module loaded.')


### 6.3 Training Step (Single Epoch)


In [ ]:
from cross_attn import train_epoch

print('Part 3: Cross-Attention single-epoch training module loaded.')


### 6.4 Training Execution with Hardcoded Arguments
Hyperparameters and execution arguments are hardcoded directly below instead of requiring a `main()` function with `argparse`.
The pipeline integrates the new data preprocessing method (verifying image integrity via `is_valid_image` to drop corrupted images, and consolidating labels to 3 classes) before DataLoader batching.


In [ ]:
from cross_attn import CrossAttentionArgs, train_loop, run_training

print('Part 4: Multi-epoch Cross-Attention training loop loaded.')


In [ ]:
from types import SimpleNamespace

# Hardcoded arguments instead of main() function / argparse
args = SimpleNamespace(
    seeds=[0, 1, 2],
    epochs=5,
    batch_size=16,
    eval_batch_size=32,
    lr_backbone=1.5e-5,
    lr_head=5e-4,
    max_length=128,
    attn_dim=256,
    num_heads=4,
    dropout=0.2,
    workers=2,
    freeze_image=True,
    data_dir=os.environ.get("DATA_DIR", "data/memotion"),
    holdout_file=os.path.join(os.environ.get("FEATURE_DIR", "features"), "train_holdout.json"),
    results_dir=os.environ.get("RESULTS_DIR", "results"),
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print("Training configuration (hardcoded arguments):")
for k, v in vars(args).items():
    print(f"  {k}: {v}")

# Execute Cross-Attention training across seeds with preprocessed data integrity checks
training_summary = run_training(args, device=device)


### 6.5 Verification & Contract Tests
Verify output contracts and schemas against `tests/test_results.py` and `tests/test_cross_attention.py`.


In [ ]:
!pytest tests/test_cross_attention.py tests/test_results.py -v


### 6.6 Results Summary
Aggregate performance metrics (Macro-F1 and Accuracy) across all seeds.


In [ ]:
import os, sys
sys.path.insert(0, 'src')
import numpy as np
import pandas as pd
from results import load_results

records = []
for s in [0, 1, 2]:
    try:
        res = load_results('both_cross_attn', s)
    except FileNotFoundError:
        continue
    records.append({
        'Seed': s,
        'Best Epoch (Holdout)': res['extra'].get('best_epoch', 'N/A'),
        'Holdout Macro-F1': round(res['extra'].get('holdout_macro_f1', 0.0), 4),
        'Test Accuracy': round(res['accuracy'], 4),
        'Test Macro-F1': round(res['macro_f1'], 4)
    })

if records:
    df = pd.DataFrame(records)
    if len(records) > 1:
        mean_row = pd.DataFrame([{
            'Seed': 'Mean ± Std',
            'Best Epoch (Holdout)': '-',
            'Holdout Macro-F1': f"{df['Holdout Macro-F1'].mean():.4f} ± {df['Holdout Macro-F1'].std():.4f}",
            'Test Accuracy': f"{df['Test Accuracy'].mean():.4f} ± {df['Test Accuracy'].std():.4f}",
            'Test Macro-F1': f"{df['Test Macro-F1'].mean():.4f} ± {df['Test Macro-F1'].std():.4f}"
        }])
        summary_df = pd.concat([df, mean_row], ignore_index=True)
    else:
        summary_df = df
    display(summary_df)
else:
    print('No results found. Run Cross-Attention training cell above first.')


In [ ]:
try:
    from google.colab import files
    for s in [0, 1, 2]:
        f_path = f"results/both_cross_attn_seed{s}.json"
        if os.path.exists(f_path):
            files.download(f_path)
except Exception:
    pass


## 7. Evaluation, Ablation & Error Analysis


## 8. Demo Inference
